# 03 — Syntactic Comparison: pandas / Dask / Koalas

This notebook fulfils the assignment requirement to **document syntactic differences**
among the libraries and to **structure the initial parts of the report notebook**.

It also loads the timing results from notebooks 01 and 02 and produces the
**combined comparison table** that will appear in the report.

Sections:
1. Library background & design philosophy
2. Side-by-side syntax comparison (all 15 operations)
3. Key syntactic differences table
4. Combined benchmark results table
5. Speedup analysis (Dask/Koalas ratio, mirroring the blog charts)

---
## 1  Library Background

### pandas
The de-facto standard for tabular data in Python.  All operations are **eager** and
run in a single process.  Limited to data that fits in RAM.

### Dask
A parallel computing library that mirrors the pandas API by splitting a DataFrame
into **partitions** and lazily building a task graph.  The graph is executed
when `.compute()` is called.  Scales from a laptop to a cluster by swapping the
scheduler (`synchronous`, `threaded`, `distributed`).

| | pandas | Dask |
|---|---|---|
| Evaluation | Eager | Lazy (task graph) |
| Scale | Single machine, RAM-bound | Multi-core / multi-node |
| API coverage | Full | ~80 % (some gaps in groupby, window) |
| Version used | 2.x | 2024.x |

### Koalas / pyspark.pandas
Implements the pandas API on top of Apache Spark.  Since PySpark 3.2 the package
is bundled as `pyspark.pandas`.
Every expression is translated to a **Spark SQL plan** and optimised by the
Catalyst query optimizer before execution.

| | Dask | Koalas (pyspark.pandas) |
|---|---|---|
| Underlying engine | Python task graph | Spark SQL / JVM |
| Query optimiser | None (Python-level) | Catalyst (filter pushdown, column pruning, …) |
| Code generation | No | Yes (Tungsten, whole-stage codegen) |
| Caching | `client.persist()` | `df.spark.cache()` |
| Force execution | `.compute()` | scalar ops auto-execute; series → `.to_pandas()` |

---
## 2  Side-by-side Syntax: all 15 operations

The table below shows equivalent code for each of the 15 operations benchmarked.

### 2a  Setup / import

In [ ]:
# --- pandas ---
import pandas as pd
# df = pd.read_parquet('yellow_taxi.parquet')

# --- Dask ---
import dask.dataframe as dd
# df = dd.read_parquet('yellow_taxi.parquet')      # lazy; .compute() triggers execution

# --- Koalas / pyspark.pandas ---
import pyspark.pandas as ps                        # replaces: import databricks.koalas as ks
# df = ps.read_parquet('yellow_taxi.parquet')      # lazy; backed by Spark plan

print('imports OK')

### 2b  Operation-by-operation comparison

In [ ]:
import numpy as np

# We print equivalent snippets side-by-side.
# (The code blocks below are illustrative — not executed against real data here.)

operations_table = [
    {
        'name': 'read_parquet',
        'pandas':  "df = pd.read_parquet(path)",
        'dask':    "df = dd.read_parquet(path)   # lazy",
        'koalas':  "df = ps.read_parquet(path)   # lazy (Spark plan)",
        'note':    'Dask/Koalas: no I/O until .compute() / action',
    },
    {
        'name': 'count',
        'pandas':  "len(df)",
        'dask':    "len(df)                       # calls __len__ → .compute()",
        'koalas':  "len(df)                       # triggers Spark COUNT action",
        'note':    'API identical; Dask/Koalas trigger a full scan',
    },
    {
        'name': 'count_index',
        'pandas':  "len(df.index)",
        'dask':    "len(df.index)",
        'koalas':  "len(df.index)",
        'note':    'Identical API; Koalas is significantly faster (Spark index opt)',
    },
    {
        'name': 'mean',
        'pandas':  "df['fare_amt'].mean()",
        'dask':    "df['fare_amt'].mean().compute()",
        'koalas':  "df['fare_amt'].mean()         # returns Python float (immediate)",
        'note':    'Dask requires .compute() for aggregations returning a scalar',
    },
    {
        'name': 'std',
        'pandas':  "df['fare_amt'].std()",
        'dask':    "df['fare_amt'].std().compute()",
        'koalas':  "df['fare_amt'].std()",
        'note':    '',
    },
    {
        'name': 'series_add',
        'pandas':  "df['fare_amt'] + df['tip_amt']",
        'dask':    "(df['fare_amt'] + df['tip_amt']).compute()",
        'koalas':  "(df['fare_amt'] + df['tip_amt']).to_pandas()",
        'note':    'Series ops return a lazy object; materialise with compute/to_pandas',
    },
    {
        'name': 'series_mul',
        'pandas':  "df['fare_amt'] * df['tip_amt']",
        'dask':    "(df['fare_amt'] * df['tip_amt']).compute()",
        'koalas':  "(df['fare_amt'] * df['tip_amt']).to_pandas()",
        'note':    '',
    },
    {
        'name': 'mean_series_add',
        'pandas':  "(df['fare_amt'] + df['tip_amt']).mean()",
        'dask':    "(df['fare_amt'] + df['tip_amt']).mean().compute()",
        'koalas':  "(df['fare_amt'] + df['tip_amt']).mean()",
        'note':    '',
    },
    {
        'name': 'mean_series_mul',
        'pandas':  "(df['fare_amt'] * df['tip_amt']).mean()",
        'dask':    "(df['fare_amt'] * df['tip_amt']).mean().compute()",
        'koalas':  "(df['fare_amt'] * df['tip_amt']).mean()",
        'note':    '',
    },
    {
        'name': 'complex_arithmetic',
        'pandas':  "np.sin(df['fare_amt']) + np.cos(df['tip_amt']) + np.arctan2(df['fare_amt'], df['tip_amt'])",
        'dask':    "(np.sin(df['fare_amt']) + np.cos(df['tip_amt']) + np.arctan2(df['fare_amt'], df['tip_amt'])).compute()",
        'koalas':  "(np.sin(df['fare_amt']) + np.cos(df['tip_amt']) + np.arctan2(df['fare_amt'], df['tip_amt'])).to_pandas()",
        'note':    'NumPy ufuncs dispatched via __array_ufunc__; syntax identical',
    },
    {
        'name': 'mean_complex_arithmetic',
        'pandas':  "(np.sin(df['fare_amt']) + ...).mean()",
        'dask':    "(np.sin(df['fare_amt']) + ...).mean().compute()",
        'koalas':  "(np.sin(df['fare_amt']) + ...).mean()",
        'note':    '',
    },
    {
        'name': 'groupby_stats',
        'pandas':  "df.groupby('vendor_name')['fare_amt'].agg(['mean', 'std'])",
        'dask':    "df.groupby('vendor_name')['fare_amt'].agg(['mean', 'std']).compute()",
        'koalas':  "df.groupby('vendor_name')['fare_amt'].agg(['mean', 'std']).to_pandas()",
        'note':    'API identical; Dask has some agg limitations compared to pandas',
    },
    {
        'name': 'join',
        'pandas':  "df.merge(df2, on='vendor_name')",
        'dask':    "df.merge(df2, on='vendor_name').compute()",
        'koalas':  "df.merge(df2, on='vendor_name').to_pandas()",
        'note':    'Koalas uses BroadcastHashJoin when df2 is small (Catalyst)',
    },
    {
        'name': 'join_count',
        'pandas':  "len(df.merge(df2, on='vendor_name'))",
        'dask':    "len(df.merge(df2, on='vendor_name'))",
        'koalas':  "len(df.merge(df2, on='vendor_name'))",
        'note':    'Koalas avoids full materialisation; counts via Spark COUNT',
    },
    {
        'name': 'value_counts',
        'pandas':  "df['payment_type'].value_counts()",
        'dask':    "df['payment_type'].value_counts().compute()",
        'koalas':  "df['payment_type'].value_counts().to_pandas()",
        'note':    '',
    },
]

df_syntax = pd.DataFrame(operations_table).set_index('name')
df_syntax

### 2c  Filtering syntax

In [ ]:
# The three libraries share IDENTICAL filter syntax:

# pandas / Dask / Koalas:
# df_filtered = df[(df['tip_amt'] >= 1) & (df['tip_amt'] < 5)]

# The difference is WHEN the filter executes:
# - pandas:  immediately (the filtered DataFrame is returned)
# - Dask:    lazily added to the task graph; executed on .compute()
# - Koalas:  lazily added to the Spark plan; executed by Catalyst with filter pushdown

print('Filtering syntax is identical across pandas / Dask / Koalas.')
print('Execution semantics differ:')
print('  pandas  → eager (immediate)')
print('  Dask    → lazy task graph (executed on .compute())')
print('  Koalas  → lazy Spark plan (Catalyst pushes filter to file scan)')

### 2d  Caching syntax

In [ ]:
print('=' * 60)
print('CACHING — Dask')
print('=' * 60)
print("""
from dask.distributed import Client, wait

client = Client()
df_filtered = df[(df['tip_amt'] >= 1) & (df['tip_amt'] < 5)]

# Persist into worker memory
df_cached = client.persist(df_filtered)
wait(df_cached)     # block until all partitions are materialised
""")

print('=' * 60)
print('CACHING — Koalas / pyspark.pandas')
print('=' * 60)
print("""
df_filtered = df[(df['tip_amt'] >= 1) & (df['tip_amt'] < 5)]

# Mark for Spark caching
df_filtered.spark.cache()

# Materialise (force execution of the plan)
df_filtered.spark.apply(lambda sdf: sdf.count())
""")

---
## 3  Key Syntactic Differences Table

In [ ]:
diff_data = {
    'Aspect': [
        'Import',
        'Read Parquet',
        'Trigger execution (scalar)',
        'Trigger execution (Series)',
        'Caching',
        'Wait for cache',
        'Distributed setup',
        'Inspect query plan',
        'Number of partitions',
    ],
    'pandas': [
        'import pandas as pd',
        'pd.read_parquet(path)',
        'automatic (eager)',
        'automatic (eager)',
        'N/A (already in RAM)',
        'N/A',
        'N/A',
        'N/A',
        'N/A',
    ],
    'Dask': [
        'import dask.dataframe as dd',
        'dd.read_parquet(path)',
        '.compute()',
        '.compute()',
        'client.persist(df)',
        'wait(df)',
        'Client(scheduler_address)',
        'df.visualize() / df.__dask_graph__()',
        'df.npartitions',
    ],
    'Koalas / pyspark.pandas': [
        'import pyspark.pandas as ps',
        'ps.read_parquet(path)',
        'automatic (Spark action)',
        '.to_pandas()',
        'df.spark.cache()',
        'df.spark.apply(lambda sdf: sdf.count())',
        'SparkSession.builder.master("yarn").getOrCreate()',
        'df.spark.explain()',
        'df.spark.frame.rdd.getNumPartitions()',
    ],
}

df_diff = pd.DataFrame(diff_data).set_index('Aspect')
df_diff

---
## 4  Combined Benchmark Results

Load the CSV files produced by notebooks 01 and 02.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import RESULTS_DIR

dask_csv   = os.path.join(RESULTS_DIR, 'dask_benchmark.csv')
koalas_csv = os.path.join(RESULTS_DIR, 'koalas_benchmark.csv')

dask_results   = pd.read_csv(dask_csv,   index_col=0) if os.path.exists(dask_csv)   else None
koalas_results = pd.read_csv(koalas_csv, index_col=0) if os.path.exists(koalas_csv) else None

if dask_results is not None:
    print('Dask results loaded:')
    print(dask_results.to_string(float_format=lambda x: f'{x:.3f}'))
else:
    print('dask_benchmark.csv not found — run notebook 01 first.')

if koalas_results is not None:
    print('\nKoalas results loaded:')
    print(koalas_results.to_string(float_format=lambda x: f'{x:.3f}'))
else:
    print('koalas_benchmark.csv not found — run notebook 02 first.')

In [ ]:
# Build the combined comparison table (rows = operations, columns = library × scenario)
if dask_results is not None and koalas_results is not None:
    combined = pd.concat(
        [dask_results.add_prefix('dask_'), koalas_results.add_prefix('koalas_')],
        axis=1
    )
    print('\nCOMBINED COMPARISON TABLE (seconds):')
    print(combined.to_string(float_format=lambda x: f'{x:.3f}'))

    out = os.path.join(RESULTS_DIR, 'combined_benchmark.csv')
    combined.to_csv(out)
    print(f'\nSaved: {out}')

---
## 5  Speedup Analysis (Dask/Koalas ratio)

Following the blog's methodology: each bar shows `Dask time / Koalas time`.
A ratio > 1 means Koalas is faster.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.benchmark_utils import geometric_mean

if dask_results is not None and koalas_results is not None:
    scenarios = [
        ('Standard',        'dask_standard',  'koalas_standard'),
        ('Filtered',        'dask_filtered',  'koalas_filtered'),
        ('Filtered+Cached', 'dask_cached',    'koalas_cached'),
    ]

    fig, axes = plt.subplots(1, len(scenarios), figsize=(18, 6), sharey=False)
    fig.suptitle('Speedup ratio: Dask time / Koalas time  (> 1 → Koalas faster)', fontsize=13)

    for ax, (title, dask_col, koalas_col) in zip(axes, scenarios):
        d = dask_results[dask_col]   if dask_col   in dask_results.columns   else None
        k = koalas_results[koalas_col] if koalas_col in koalas_results.columns else None

        if d is None or k is None:
            ax.set_title(f'{title}\n(data missing)')
            continue

        ratio = (d / k).dropna()
        colors = ['#e74c3c' if v > 1 else '#2ecc71' for v in ratio.values]
        ratio.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
        ax.axhline(y=1, color='black', linewidth=0.8, linestyle='--')

        gm = geometric_mean(ratio.dropna().tolist())
        ax.set_title(f'{title}\nGeometric mean: {gm:.2f}x')
        ax.set_ylabel('Dask time / Koalas time')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'speedup_comparison.png'), dpi=120)
    plt.show()
else:
    print('Run notebooks 01 and 02 to generate results first.')

---
## 6  Summary Table for Report

This is the table that should appear in the report (Experiment #1).

Rows = operations, Columns = library × scenario, Cells = elapsed time in seconds.

In [ ]:
if dask_results is not None and koalas_results is not None:
    # Rename columns for readability
    rename_map = {
        'dask_standard':   'Dask / Standard',
        'dask_filtered':   'Dask / Filter',
        'dask_cached':     'Dask / Filter+Cache',
        'koalas_standard': 'Koalas / Standard',
        'koalas_filtered': 'Koalas / Filter',
        'koalas_cached':   'Koalas / Filter+Cache',
    }
    report_table = combined.rename(columns=rename_map)
    print('EXPERIMENT 1 — Execution Times (seconds)')
    print(report_table.to_string(float_format=lambda x: f'{x:.3f}'))
else:
    print('Placeholder — run notebooks 01 and 02 to populate.')